Question 3.1 Compute the six measures in every frame and the five statistics for every video, obtaining a table of 
30 features with one row per video. Check that it has no missing values. The same functions compute 
the features of the test videos, which are set aside without being inspected.

In [ ]:
import pandas as pd
import numpy as np
import math

og_df = pd.read_parquet("t1.1.gzip")
d = np.sqrt((og_df["x_45"] - og_df["x_36"])**2 + (og_df["y_45"] - og_df["y_36"])**2)
cols = {}

cols["video"] = og_df["video"]
cols["frame"] = og_df["frame"]
for k in range(68):
    cols[f"x_{k}"] = (og_df[f"x_{k}"] - og_df["x_27"]) / d
    cols[f"y_{k}"] = (og_df[f"y_{k}"] - og_df["y_27"]) / d

normalized_df = pd.DataFrame(cols)
normalized_df.to_parquet("normalized_data.gzip", compression="gzip")

def distance(df, a, b):
    return np.sqrt(
        (df[f"x_{a}"] - df[f"x_{b}"])**2
        + (df[f"y_{a}"] - df[f"y_{b}"])**2
    )

measurements = {}
measurements["video"] = normalized_df["video"]
measurements["frame"] = normalized_df["frame"]
#Mouth opening - δ(62, 66)
measurements["M1"]= distance(normalized_df, 66, 62)
#Mouth width - δ(48, 54)
measurements["M2"]= distance(normalized_df, 54, 48)
#Corner lift - mean y of 51 and 57 minus mean y of 48 and 54;positive when the corners of the mouth are above the centre of the lips
measurements["M3"]= ((normalized_df["y_51"] + normalized_df["y_57"]) / 2) - ((normalized_df["y_48"] + normalized_df["y_54"]) / 2)
#Brow height - mean y of 36–41 minus mean y of 17–21, averaged with mean y of 42–47 minus mean y of 22–26 (eye centroid minus brow centroid, both sides)
measurements["M4"]= (
    (
        (normalized_df["y_36"] + normalized_df["y_37"] + normalized_df["y_38"] + normalized_df["y_39"] + normalized_df["y_40"] + normalized_df["y_41"]) / 6
    ) - (
        (normalized_df["y_17"] + normalized_df["y_18"] + normalized_df["y_19"] + normalized_df["y_20"] + normalized_df["y_21"]) / 5
    ) + (
        (normalized_df["y_42"] + normalized_df["y_43"] + normalized_df["y_44"] + normalized_df["y_45"] + normalized_df["y_46"] + normalized_df["y_47"]) / 6
    ) - (
        (normalized_df["y_22"] + normalized_df["y_23"] + normalized_df["y_24"] + normalized_df["y_25"] + normalized_df["y_26"]) / 5
    )
) / 2
#Eye opening ¼ [δ(37, 41) + δ(38, 40) + δ(43, 47) + δ(44, 46)]
measurements["M5"]= (
    distance(normalized_df, 37, 41) +
    distance(normalized_df, 38, 40) +
    distance(normalized_df, 43, 47) +
    distance(normalized_df, 44, 46)
) / 4
#Brow gap δ(21, 22)
measurements["M6"]= distance(normalized_df, 21, 22)

per_frame_measurements_df = pd.DataFrame(measurements)

video_measurements = []
for video, group in per_frame_measurements_df.groupby("video"):
    row = {"video": video}
    group = group.sort_values("frame")
    for i in ["M1", "M2", "M3", "M4", "M5", "M6"]:
        row[f"{i}_mean"] = group[i].mean()
        row[f"{i}_std"] = group[i].std()
        row[f"{i}_range"] = group[i].max() - group[i].min()
        #mean absolute velocity, v - the mean of |m(t + 1) − m(t)| over the T − 1 pairs of consecutive frames
        row[f"{i}_v"] = (group[i].diff().abs().sum() / (len(group) - 1)) if len(group) > 1 else 0
        #movement energy, E - the mean of (m(t + 1) − m(t))² over the T − 1 pairs of consecutive frames
        row[f"{i}_E"] = (group[i].diff().pow(2).sum() / (len(group) - 1)) if len(group) > 1 else 0
    video_measurements.append(row)

video_measurements_df = pd.DataFrame(video_measurements)
#check for missing values and print the number of missing values for each column
missing_values = video_measurements_df.isnull().sum()
print(f"Missing values for each column:\n{missing_values}")

video_measurements_df.to_parquet("video_measurements.gzip", compression="gzip")


Missing values for each column:
video       0
M1_mean     0
M1_std      0
M1_range    0
M1_v        0
M1_E        0
M2_mean     0
M2_std      0
M2_range    0
M2_v        0
M2_E        0
M3_mean     0
M3_std      0
M3_range    0
M3_v        0
M3_E        0
M4_mean     0
M4_std      0
M4_range    0
M4_v        0
M4_E        0
M5_mean     0
M5_std      0
M5_range    0
M5_v        0
M5_E        0
M6_mean     0
M6_std      0
M6_range    0
M6_v        0
M6_E        0
dtype: int64
                         video   M1_mean    M1_std  M1_range      M1_v  \
1     01-01-01-01-01-01-02.csv  0.045541  0.042978  0.166364  0.007619   
2     01-01-01-01-01-01-03.csv  0.044769  0.030441  0.120792  0.006470   
3     01-01-01-01-01-01-04.csv  0.046652  0.045070  0.216453  0.007799   
4     01-01-01-01-01-01-05.csv  0.038562  0.022994  0.086473  0.004317   
5     01-01-01-01-01-01-06.csv  0.093865  0.059834  0.254196  0.010456   
...                        ...       ...       ...       ...       ...   
143

In [10]:

testIndex = pd.read_parquet("testIndex.gzip")
print(video_measurements_df[~video_measurements_df["video"].isin(testIndex["video"])])

                         video   M1_mean    M1_std  M1_range      M1_v  \
1     01-01-01-01-01-01-02.csv  0.045541  0.042978  0.166364  0.007619   
2     01-01-01-01-01-01-03.csv  0.044769  0.030441  0.120792  0.006470   
3     01-01-01-01-01-01-04.csv  0.046652  0.045070  0.216453  0.007799   
4     01-01-01-01-01-01-05.csv  0.038562  0.022994  0.086473  0.004317   
5     01-01-01-01-01-01-06.csv  0.093865  0.059834  0.254196  0.010456   
...                        ...       ...       ...       ...       ...   
1434  01-01-08-02-02-02-19.csv  0.099246  0.029073  0.166773  0.008806   
1435  01-01-08-02-02-02-20.csv  0.051768  0.032016  0.104523  0.004976   
1436  01-01-08-02-02-02-21.csv  0.148165  0.057368  0.269216  0.015000   
1437  01-01-08-02-02-02-22.csv  0.073067  0.034145  0.136747  0.007584   
1438  01-01-08-02-02-02-23.csv  0.033270  0.030631  0.111538  0.006106   

          M1_E   M2_mean    M2_std  M2_range      M2_v  ...   M5_mean  \
1     0.000202  0.572101  0.016066  0.

Question 3.2 Propose and compute at least five features of your own, each with a sentence of justification stating 
the expression it targets and the landmarks it uses. Only information from the landmark files may be 
used: the fields of the file name other than the emotion (actor, intensity, statement, repetition) are 
not features.